In [6]:
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
import joblib

# Load the trained model and scaler
model = joblib.load('pose_verification_model.pkl')
scaler = joblib.load('scaler.pkl')

# Load the image
image_path = 'C:/Users/HP/Desktop/Dataset/Downdog/00000024.jpg'
image = cv2.imread(image_path)
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Initialize MediaPipe pose estimation
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=True, model_complexity=2)

# Perform pose detection
results = pose.process(image_rgb)

# Extract keypoints
keypoints = results.pose_landmarks.landmark if results.pose_landmarks else None

# Close the MediaPipe instance
pose.close()

if keypoints:
    def calculate_angle(a, b, c):
        a = np.array(a)  # First point
        b = np.array(b)  # Middle point
        c = np.array(c)  # End point
        
        radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(a[1] - b[1], a[0] - b[0])
        angle = np.abs(radians * 180.0 / np.pi)
        
        if angle > 180.0:
            angle = 360 - angle
            
        return angle

    # Extract the necessary keypoints for angle calculation
    left_shoulder = [keypoints[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x,
                     keypoints[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y]
    left_elbow = [keypoints[mp_pose.PoseLandmark.LEFT_ELBOW.value].x,
                  keypoints[mp_pose.PoseLandmark.LEFT_ELBOW.value].y]
    left_wrist = [keypoints[mp_pose.PoseLandmark.LEFT_WRIST.value].x,
                  keypoints[mp_pose.PoseLandmark.LEFT_WRIST.value].y]

    right_shoulder = [keypoints[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].x,
                      keypoints[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y]
    right_elbow = [keypoints[mp_pose.PoseLandmark.RIGHT_ELBOW.value].x,
                   keypoints[mp_pose.PoseLandmark.RIGHT_ELBOW.value].y]
    right_wrist = [keypoints[mp_pose.PoseLandmark.RIGHT_WRIST.value].x,
                   keypoints[mp_pose.PoseLandmark.RIGHT_WRIST.value].y]

    # Calculate the angles
    left_elbow_angle = calculate_angle(left_shoulder, left_elbow, left_wrist)
    right_elbow_angle = calculate_angle(right_shoulder, right_elbow, right_wrist)
    left_shoulder_angle = calculate_angle(left_elbow, left_shoulder, [left_shoulder[0], left_shoulder[1] - 1])
    right_shoulder_angle = calculate_angle(right_elbow, right_shoulder, [right_shoulder[0], right_shoulder[1] - 1])
    
    left_knee = [keypoints[mp_pose.PoseLandmark.LEFT_KNEE.value].x,
                 keypoints[mp_pose.PoseLandmark.LEFT_KNEE.value].y]
    left_hip = [keypoints[mp_pose.PoseLandmark.LEFT_HIP.value].x,
                keypoints[mp_pose.PoseLandmark.LEFT_HIP.value].y]
    left_ankle = [keypoints[mp_pose.PoseLandmark.LEFT_ANKLE.value].x,
                  keypoints[mp_pose.PoseLandmark.LEFT_ANKLE.value].y]

    right_knee = [keypoints[mp_pose.PoseLandmark.RIGHT_KNEE.value].x,
                  keypoints[mp_pose.PoseLandmark.RIGHT_KNEE.value].y]
    right_hip = [keypoints[mp_pose.PoseLandmark.RIGHT_HIP.value].x,
                 keypoints[mp_pose.PoseLandmark.RIGHT_HIP.value].y]
    right_ankle = [keypoints[mp_pose.PoseLandmark.RIGHT_ANKLE.value].x,
                   keypoints[mp_pose.PoseLandmark.RIGHT_ANKLE.value].y]

    left_knee_angle = calculate_angle(left_hip, left_knee, left_ankle)
    right_knee_angle = calculate_angle(right_hip, right_knee, right_ankle)

    # Create a DataFrame for the angles
    angles_df = pd.DataFrame([{
        'left_elbow_angle': left_elbow_angle,
        'right_elbow_angle': right_elbow_angle,
        'left_shoulder_angle': left_shoulder_angle,
        'right_shoulder_angle': right_shoulder_angle,
        'left_knee_angle': left_knee_angle,
        'right_knee_angle': right_knee_angle
    }])

    # Ensure the columns are in the correct order
    angles_df = angles_df[['left_elbow_angle', 'right_elbow_angle', 'left_shoulder_angle',
                           'right_shoulder_angle', 'left_knee_angle', 'right_knee_angle']]

    # Scale the angles using the previously fitted scaler
    angles_scaled = scaler.transform(angles_df)

    # Predict using the trained model
    pose_correctness = model.predict(angles_scaled)
    print(f"Pose Correctness: {'Correct' if pose_correctness == 1 else 'Incorrect'}")
else:
    print("No keypoints detected")


Pose Correctness: Incorrect


C:\Users\HP\anaconda3\envs\mediapipe\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
